## Model Configuration

In [142]:
config = {
    "datasets": {
        "dti": {
            "path": "../../Data/scope_onside_common_v3.parquet",   
            "drug_id_col": "drug_chembl_id",
            "prot_id_col": "target_uniprot_id",
            "label_col": "label",
            "rxcui_col": "rxcui"
        },
        "adr": {
            "path": "../../Data/final_rxnorm_meddra_v2.parquet",          
            "rxcui_col": "rxnorm_ingredient_id",
            "meddra_id_col": "meddra_id",
            "meddra_name_col": "meddra_name"        
        }
    },

    "embeddings": {
        "protein": [
            {
                "name": "esm", 
                "path": "../../Data/3. Protein_enbeddings/ESM_embeddings_(t33_650m model).parquet", 
                "id_col": "id", 
                "emb_col": "embedding"
            },
            {
                "name": "gvp_gnn", 
                "path": "../../Data/3. Protein_enbeddings/GVP-GNN_protein_embeddings.parquet", 
                "id_col": "uniprot_id", 
                "emb_col": "embedding"
            },
        ],

        "drug": [
            {
                "name": "egnn", 
                "path": "../../Data/2. Drug_embeddings/EGNN_drug_embeddings_v2.parquet", 
                "id_col": "drug_chembl_id", 
                "emb_col": "embedding"
            },
            {
                "name": "chemberta", 
                "path": "../../Data/2. Drug_embeddings/smiles_embeddings_chemberta.parquet", 
                "id_col": "drug_chembl_id", 
                "emb_col": "embedding"
            },
        ]
    },

    "fusion": {
        # method: "concat" | "mean" | "gated"
        "protein": {
            "enabled": True, 
            "method": "gated", 
            "embed": ["esm", "gvp_gnn"],

            "gated": {
                "proj_dim": 512,                 # project each source to same dim
                "gate_hidden": 256,              # MLP hidden size for gate net
                "dropout": 0.1,
                "activation": "gelu",            # "relu" | "gelu" | "silu"
                "temperature": 1.0,              # softmax temperature for gate weights
                "use_layernorm": True,
                "gating": "softmax",             # "softmax" (sums to 1) | "sigmoid" (independent)
                "residual": True                 # add mean(proj_sources) as residual stabilizer
            }
        },
        "drug": {
            "enabled": True, 
            "method": "gated", 
            "embed": ["chemberta", "egnn"],

            "gated": {
                "proj_dim": 512,
                "gate_hidden": 256,
                "dropout": 0.1,
                "activation": "gelu",
                "temperature": 1.0,
                "use_layernorm": True,
                "gating": "softmax",
                "residual": True
            }
        }
    },

    "model": {
        "latent_dim": 512
    },

    "dataloader": {
        "batch_size": 512,
    },

    "runtime": {
        "device": "cuda",
        "seed": 42,

    }
}


In [143]:
import warnings
warnings.filterwarnings("ignore")

## Data Loading 

### Dataset loading

In [144]:
from dataclasses import dataclass
from typing import Dict, Any, List
import pandas as pd

class IDataSource:
    def load(self) -> pd.DataFrame:
        raise NotImplementedError

@dataclass
class ParquetDataSource(IDataSource):
    path: str

    def load(self) -> pd.DataFrame:
        return pd.read_parquet(self.path).reset_index(drop=True)

def _require_cols(df: pd.DataFrame, cols: List[str], name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] Missing required columns: {missing}. Found: {list(df.columns)}")

class DatasetRegistry:
    def __init__(self, config: Dict[str, Any]):
        self.cfg = config
        self.dti_df = None
        self.adr_df = None

    def load_all(self) -> Dict[str, pd.DataFrame]:
        dti_cfg = self.cfg["datasets"]["dti"]
        adr_cfg = self.cfg["datasets"]["adr"]

        self.dti_df = ParquetDataSource(dti_cfg["path"]).load()
        self.adr_df = ParquetDataSource(adr_cfg["path"]).load()

        _require_cols(self.dti_df, [dti_cfg["drug_id_col"], dti_cfg["prot_id_col"], dti_cfg["label_col"]], "DTI")
        _require_cols(self.adr_df, [adr_cfg["rxcui_col"], adr_cfg["meddra_id_col"]], "ADR")

        self._normalize()
        return {"dti": self.dti_df, "adr": self.adr_df}

    def _normalize(self):
        dti_cfg = self.cfg["datasets"]["dti"]
        adr_cfg = self.cfg["datasets"]["adr"]

        self.dti_df[dti_cfg["drug_id_col"]] = self.dti_df[dti_cfg["drug_id_col"]].astype(str)
        self.dti_df[dti_cfg["prot_id_col"]] = self.dti_df[dti_cfg["prot_id_col"]].astype(str)
        self.dti_df[dti_cfg["label_col"]] = self.dti_df[dti_cfg["label_col"]].astype(int)

        if dti_cfg.get("rxcui_col") in self.dti_df.columns:
            self.dti_df[dti_cfg["rxcui_col"]] = self.dti_df[dti_cfg["rxcui_col"]].astype(str)

        self.adr_df[adr_cfg["rxcui_col"]] = self.adr_df[adr_cfg["rxcui_col"]].astype(str)
        self.adr_df[adr_cfg["meddra_id_col"]] = self.adr_df[adr_cfg["meddra_id_col"]].astype(str)

        if adr_cfg.get("meddra_name_col") in self.adr_df.columns:
            self.adr_df[adr_cfg["meddra_name_col"]] = self.adr_df[adr_cfg["meddra_name_col"]].astype(str)


### Embedding loading

In [145]:
from dataclasses import dataclass
from typing import Dict, Any, Optional
import numpy as np
import pandas as pd
import torch

class IEmbeddingStore:
    @property
    def dim(self) -> int:
        raise NotImplementedError

    def has(self, _id: str) -> bool:
        raise NotImplementedError

    def get_np(self, _id: str) -> np.ndarray:
        raise NotImplementedError

    def get_torch(self, _id: str, device: Optional[str] = None) -> torch.Tensor:
        arr = self.get_np(_id)
        t = torch.from_numpy(arr)
        return t.to(device) if device else t

@dataclass
class ParquetEmbeddingStore(IEmbeddingStore):
    path: str
    id_col: str
    emb_col: str = "embedding"
    dtype: Any = np.float32

    def __post_init__(self):
        df = pd.read_parquet(self.path).reset_index(drop=True)

        if self.id_col not in df.columns:
            raise ValueError(f"[EmbeddingStore] Missing id_col='{self.id_col}' in {self.path}. Found: {list(df.columns)}")
        if self.emb_col not in df.columns:
            raise ValueError(f"[EmbeddingStore] Missing emb_col='{self.emb_col}' in {self.path}. Found: {list(df.columns)}")

        df[self.id_col] = df[self.id_col].astype(str)

        store: Dict[str, np.ndarray] = {}
        for _id, emb in zip(df[self.id_col].tolist(), df[self.emb_col].tolist()):
            store[_id] = np.asarray(emb, dtype=self.dtype)

        if len(store) == 0:
            raise ValueError(f"[EmbeddingStore] No rows loaded from {self.path}")

        self._store = store
        self._dim = int(next(iter(store.values())).shape[-1])

    @property
    def dim(self) -> int:
        return self._dim

    def has(self, _id: str) -> bool:
        return str(_id) in self._store

    def get_np(self, _id: str) -> np.ndarray:
        key = str(_id)
        if key not in self._store:
            raise KeyError(f"[EmbeddingStore] ID not found: {key}")
        return self._store[key]


In [146]:
class EmbeddingRegistry:
    """
    Loads embedding sources declared in config into:
      self.protein[name] -> ParquetEmbeddingStore
      self.drug[name] -> ParquetEmbeddingStore
    """
    def __init__(self, config: Dict[str, Any]):
        self.cfg = config
        self.protein: Dict[str, ParquetEmbeddingStore] = {}
        self.drug: Dict[str, ParquetEmbeddingStore] = {}

    def load_all(self):
        emb_cfg = self.cfg["embeddings"]

        # Protein sources
        for src in emb_cfg.get("protein", []):
            name = src["name"]
            self.protein[name] = ParquetEmbeddingStore(
                path=src["path"],
                id_col=src["id_col"],
                emb_col=src.get("emb_col", "embedding"),
            )

        # Drug sources
        for src in emb_cfg.get("drug", []):
            name = src["name"]
            self.drug[name] = ParquetEmbeddingStore(
                path=src["path"],
                id_col=src["id_col"],
                emb_col=src.get("emb_col", "embedding"),
            )

        if len(self.protein) == 0:
            raise ValueError("No protein embeddings found in config['embeddings']['protein']")
        if len(self.drug) == 0:
            raise ValueError("No drug embeddings found in config['embeddings']['drug']")

        return {"protein": self.protein, "drug": self.drug}


In [147]:
from torch.utils.data import Dataset
import torch

class DTIDataset(Dataset):
    def __init__(self, dti_df, cfg, emb_reg):
        self.df = dti_df.reset_index(drop=True)
        self.cfg = cfg
        self.emb_reg = emb_reg

        dti_cfg = cfg["datasets"]["dti"]
        self.drug_col = dti_cfg["drug_id_col"]
        self.prot_col = dti_cfg["prot_id_col"]
        self.label_col = dti_cfg["label_col"]

        # which embedding sources to use (from fusion config)
        self.drug_sources = cfg["fusion"]["drug"]["embed"]
        self.prot_sources = cfg["fusion"]["protein"]["embed"]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        d_id = str(row[self.drug_col])
        p_id = str(row[self.prot_col])
        y = float(row[self.label_col])

        drug_vecs = [ torch.tensor(self.emb_reg.drug[name].get_np(d_id), dtype=torch.float32) for name in self.drug_sources]
        prot_vecs = [torch.tensor(self.emb_reg.protein[name].get_np(p_id), dtype=torch.float32) for name in self.prot_sources]

        return {
            "drug_id": d_id,
            "prot_id": p_id,
            "drug_vecs": drug_vecs,   # list[np.ndarray]
            "prot_vecs": prot_vecs,   # list[np.ndarray]
            "label": y,               # float
        }


In [148]:
def collate_dti(batch):
    # batch is list of dicts
    out = {}
    out["drug_id"] = [b["drug_id"] for b in batch]
    out["prot_id"] = [b["prot_id"] for b in batch]
    out["label"] = torch.tensor([b["label"] for b in batch], dtype=torch.float32)

    nd = len(batch[0]["drug_vecs"])
    np_ = len(batch[0]["prot_vecs"])

    out["drug_vecs"] = []
    for i in range(nd):
        arr = np.stack([b["drug_vecs"][i] for b in batch], axis=0)  # [B, d_i]
        out["drug_vecs"].append(torch.from_numpy(arr))              # float32 already

    out["prot_vecs"] = []
    for i in range(np_):
        arr = np.stack([b["prot_vecs"][i] for b in batch], axis=0)
        out["prot_vecs"].append(torch.from_numpy(arr))

    return out


## Fusion Layer

In [149]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def _act(name: str):
    name = name.lower()
    if name == "relu": return nn.ReLU()
    if name == "gelu": return nn.GELU()
    if name == "silu": return nn.SiLU()
    raise ValueError(f"Unknown activation: {name}")

class FusionModule(nn.Module):
    def __init__(self, method: str, in_dims: list[int], cfg: dict | None = None):
        super().__init__()
        self.method = method.lower()
        self.in_dims = in_dims
        self.total_dim = sum(in_dims)

        if self.method == "concat":
            self.out_dim = int(sum(in_dims))
            return

        if self.method == "mean":
            if len(set(in_dims)) != 1:
                raise ValueError(f"mean fusion requires equal dims, got {in_dims}")
            self.out_dim = int(in_dims[0])
            return

        if self.method != "gated":
            raise ValueError("method must be concat | mean | gated")

        if cfg is None:
            raise ValueError("gated fusion requires cfg")

        


        # 2) gate net produces one logit per source (per sample)
        self.gate = nn.Sequential(
            nn.Linear(self.total_dim, cfg.get("proj_dim")),
            nn.Sigmoid()
        )

        self.output_layer= nn.Sequential(
            nn.Linear(self.total_dim, cfg.get("proj_dim")),
            nn.ReLU(),
            nn.Dropout(cfg.get("dropout"))
        )

    def forward(self, xs: list[torch.Tensor]) -> torch.Tensor:
        # xs: list of [B, di]
        if self.method == "concat":
            return torch.cat(xs, dim=-1)

        if self.method == "mean":
            return torch.stack(xs, dim=0).mean(dim=0)

        combined = torch.cat(xs, dim=-1)
        g = self.gate(combined)
        f = self.output_layer(combined)


        return g * f


## Model

In [150]:
import torch.nn as nn
import torch
import torch.nn.functional as F

class DTIScorer(nn.Module):
    def __init__(self, cfg, drug_in_dims, prot_in_dims):
        super().__init__()
        fcfg_d = cfg["fusion"]["drug"]
        fcfg_p = cfg["fusion"]["protein"]

        self.drug_fusion = FusionModule(
            method=fcfg_d["method"],
            in_dims=drug_in_dims,
            cfg=fcfg_d.get("gated") if fcfg_d["method"] == "gated" else None
        )
        self.prot_fusion = FusionModule(
            method=fcfg_p["method"],
            in_dims=prot_in_dims,
            cfg=fcfg_p.get("gated") if fcfg_p["method"] == "gated" else None
        )

        self.protein_projector = nn.Sequential(
            nn.Linear(config["model"]["latent_dim"], config["model"]["latent_dim"]),
            nn.BatchNorm1d(config["model"]["latent_dim"]),
            nn.ReLU(),
            nn.Linear(config["model"]["latent_dim"], config["model"]["latent_dim"])
        )
        
        self.drug_projector = nn.Sequential(
            nn.Linear(config["model"]["latent_dim"], config["model"]["latent_dim"]),
            nn.BatchNorm1d(config["model"]["latent_dim"]),
            nn.ReLU(),
            nn.Linear(config["model"]["latent_dim"], config["model"]["latent_dim"])
        )


    def forward(self, drug_vecs, prot_vecs):
        # drug_vecs: list of [B, d_i], prot_vecs: list of [B, p_i]
        p_fused = self.prot_fusion(prot_vecs)
        z_p = self.protein_projector(p_fused)


        d_fused = self.drug_fusion(drug_vecs)
        z_d = self.drug_projector(d_fused)


        z_p = F.normalize(z_p, p=2, dim=-1)
        z_d = F.normalize(z_d, p=2, dim=-1)
        
        return z_p, z_d


In [151]:
class MultiTaskLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
        self.mse = nn.MSELoss()

    def dti_contrastive_loss(self, z_p, z_d):
        """
        Standard InfoNCE loss.
        z_p: Protein embeddings [batch, latent_dim]
        z_d: Drug embeddings [batch, latent_dim]
        """
        batch_size = z_p.shape[0]
        
        # Compute similarity matrix (Cosine similarity because vectors are normalized)
        logits = torch.matmul(z_d, z_p.T) / self.temperature
        
        # Ground truth is the diagonal (each drug matches its own protein in the batch)
        labels = torch.arange(batch_size).to(z_p.device)
        
        loss_p = F.cross_entropy(logits, labels)
        loss_d = F.cross_entropy(logits.T, labels)
        
        return (loss_p + loss_d) / 2


class DTISupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07, pos_weight=1.5):
        super().__init__()
        self.temperature = temperature
        self.pos_weight = float(pos_weight)

    def forward(self, z_p, z_d, y):
        """
        z_p, z_d: [B, latent_dim] (normalized)
        y: [B] float 0/1
        """
        # similarity logit (scaled)
        logits = (z_d * z_p).sum(dim=-1) / self.temperature  # [B]

        # Weighted BCE: pos_weight > 1 penalizes missing positives more
        loss = F.binary_cross_entropy_with_logits(
            logits, y, pos_weight=torch.tensor(self.pos_weight, device=logits.device)
        )
        return loss


## Train Test Split

In [152]:
import numpy as np

def drug_level_split(dti_df, drug_col, seed=42, train=0.8, val=0.1, test=0.1):
    assert abs(train + val + test - 1.0) < 1e-6
    rng = np.random.default_rng(seed)

    drugs = dti_df[drug_col].astype(str).unique()
    rng.shuffle(drugs)

    n = len(drugs)
    n_train = int(n * train)
    n_val = int(n * val)

    train_drugs = set(drugs[:n_train])
    val_drugs   = set(drugs[n_train:n_train+n_val])
    test_drugs  = set(drugs[n_train+n_val:])

    train_df = dti_df[dti_df[drug_col].astype(str).isin(train_drugs)].reset_index(drop=True)
    val_df   = dti_df[dti_df[drug_col].astype(str).isin(val_drugs)].reset_index(drop=True)
    test_df  = dti_df[dti_df[drug_col].astype(str).isin(test_drugs)].reset_index(drop=True)

    return train_df, val_df, test_df


In [153]:
from torch.utils.data import Dataset
import torch
import pandas as pd
import numpy as np

class NegativeSamplingDTIDataset(Dataset):
    """
    Produces a mix of:
      - all positive pairs
      - sampled negative pairs per positive (ratio k)
    """
    def __init__(self, dti_df, cfg, emb_reg, neg_per_pos=1, seed=42):
        self.cfg = cfg
        self.emb_reg = emb_reg
        self.neg_per_pos = int(neg_per_pos)
        self.rng = np.random.default_rng(seed)

        dti_cfg = cfg["datasets"]["dti"]
        self.drug_col = dti_cfg["drug_id_col"]
        self.prot_col = dti_cfg["prot_id_col"]
        self.label_col = dti_cfg["label_col"]

        # keep only rows that have embeddings for selected sources (avoid runtime KeyErrors)
        self.drug_sources = cfg["fusion"]["drug"]["embed"]
        self.prot_sources = cfg["fusion"]["protein"]["embed"]

        df = dti_df.copy()
        df[self.drug_col] = df[self.drug_col].astype(str)
        df[self.prot_col] = df[self.prot_col].astype(str)
        df[self.label_col] = df[self.label_col].astype(int)

        # Positive set per drug
        pos_df = df[df[self.label_col] == 1].reset_index(drop=True)
        self.pos_df = pos_df

        # Protein universe (from whole df)
        self.all_prots = df[self.prot_col].dropna().unique().astype(str)

        # drug -> set(pos proteins)
        self.pos_map = {}
        for d, grp in pos_df.groupby(self.drug_col):
            self.pos_map[d] = set(grp[self.prot_col].tolist())

    def __len__(self):
        # each positive yields (1 + neg_per_pos) samples
        return len(self.pos_df) * (1 + self.neg_per_pos)

    def _sample_negative_protein(self, drug_id: str) -> str:
        pos_set = self.pos_map.get(drug_id, set())
        while True:
            p = str(self.rng.choice(self.all_prots))
            if p not in pos_set:
                return p

    def __getitem__(self, idx):
        pos_idx = idx // (1 + self.neg_per_pos)
        offset  = idx % (1 + self.neg_per_pos)

        row = self.pos_df.iloc[pos_idx]
        d_id = str(row[self.drug_col])

        if offset == 0:
            # positive
            p_id = str(row[self.prot_col])
            y = 1.0
        else:
            # sampled negative
            p_id = self._sample_negative_protein(d_id)
            y = 0.0

        # build per-source vectors (same format as before)
        drug_vecs = [ torch.tensor(self.emb_reg.drug[name].get_np(d_id), dtype=torch.float32) for name in self.drug_sources]   # list[np.ndarray]
        prot_vecs = [ torch.tensor(self.emb_reg.protein[name].get_np(p_id), dtype= torch.float32) for name in self.prot_sources]
        
        

        return {
            "drug_id": d_id,
            "prot_id": p_id,
            "drug_vecs": drug_vecs,
            "prot_vecs": prot_vecs,
            "label": y,   # float
        }


## Training Functions

In [154]:
import torch
import torch.nn.functional as F

def compute_pos_weight(dti_df, label_col):
    vc = dti_df[label_col].value_counts()
    pos = float(vc.get(1, 1.0))
    neg = float(vc.get(0, 1.0))
    return neg / max(pos, 1.0)

def train_one_epoch(model, loader, optimizer, device, criterion, grad_clip=1.0, use_amp=True):
    model.train()
    total, n = 0.0, 0
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    for batch in loader:
        drug_vecs = [t.to(device, non_blocking=True) for t in batch["drug_vecs"]]
        prot_vecs = [t.to(device, non_blocking=True) for t in batch["prot_vecs"]]
        y = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad()

        
        z_p, z_d = model(drug_vecs, prot_vecs)
        loss = criterion(z_p, z_d, y)

        loss.backward()
       
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        bs = y.size(0)
        total += float(loss.item()) * bs
        n += bs

    return total / max(n, 1)



In [155]:
from sklearn.metrics import roc_auc_score
import numpy as np

@torch.no_grad()
def eval_aucroc_explicit(model, loader, device):
    model.eval()
    scores_all, labels_all = [], []

    for batch in loader:
        # 1. Move data to device
        drug_vecs = [t.to(device) for t in batch["drug_vecs"]]
        prot_vecs = [t.to(device) for t in batch["prot_vecs"]]
        # Ground truth labels (1 for interaction, 0 for no interaction)
        y = batch["label"].to(device) 

        # 2. Forward pass
        z_p, z_d = model(drug_vecs, prot_vecs)

        # 3. Calculate Cosine Similarity
        z_p_norm = F.normalize(z_p, p=2, dim=-1)
        z_d_norm = F.normalize(z_d, p=2, dim=-1)
        
        # We only care about the similarity of the SPECIFIC drug/protein 
        # pairs provided in this batch, not the whole matrix.
        # This is a row-wise dot product:
        scores = torch.sum(z_d_norm * z_p_norm, dim=-1)

        # 4. Store results
        scores_all.append(scores.cpu().numpy())
        labels_all.append(y.cpu().numpy())

    # Flatten the lists into single arrays
    scores_all = np.concatenate(scores_all)
    labels_all = np.concatenate(labels_all)

    # 5. Calculate ROC-AUC
    if len(np.unique(labels_all)) < 2:
        return float("nan") # Requires both classes (0 and 1) to be present
        
    return roc_auc_score(labels_all, scores_all)


## Runtime

In [156]:
registry = DatasetRegistry(config)
dfs = registry.load_all()

dti_df = dfs["dti"]
adr_df = dfs["adr"]

print("DTI shape:", dti_df.shape)
print("ADR shape:", adr_df.shape)

print("DTI columns:", dti_df.columns.tolist())
print("ADR columns:", adr_df.columns.tolist())

print("DTI label counts:\n", dti_df[config["datasets"]["dti"]["label_col"]].value_counts())


DTI shape: (34741, 7)
ADR shape: (69474, 3)
DTI columns: ['drug_chembl_id', 'target_uniprot_id', 'label', 'smiles', 'sequence', 'molfile_3d', 'rxcui']
ADR columns: ['rxnorm_ingredient_id', 'meddra_id', 'meddra_name']
DTI label counts:
 label
0    22507
1    12234
Name: count, dtype: int64


In [157]:
emb_reg = EmbeddingRegistry(config)
emb_reg.load_all()

print("Protein sources:", {k: v.dim for k, v in emb_reg.protein.items()})
print("Drug sources:", {k: v.dim for k, v in emb_reg.drug.items()})


Protein sources: {'esm': 1280, 'gvp_gnn': 1024}
Drug sources: {'egnn': 256, 'chemberta': 384}


In [158]:
# assumes you already loaded dti_df via DatasetRegistry
dti_cfg = config["datasets"]["dti"]

drug_ids = set(dti_df[dti_cfg["drug_id_col"]].astype(str).unique())
prot_ids = set(dti_df[dti_cfg["prot_id_col"]].astype(str).unique())

# pick the first configured embedding name (or your fusion list later)
drug_store = emb_reg.drug[config["fusion"]["drug"]["embed"][0]]
prot_store = emb_reg.protein[config["fusion"]["protein"]["embed"][0]]

missing_drugs = [x for x in list(drug_ids)[:1000] if not drug_store.has(x)]
missing_prots = [x for x in list(prot_ids)[:1000] if not prot_store.has(x)]

print("Missing drugs (sample):", len(missing_drugs))
print("Missing prots (sample):", len(missing_prots))


Missing drugs (sample): 0
Missing prots (sample): 0


In [159]:
from sklearn.model_selection import train_test_split


dti_cfg = config["datasets"]["dti"]

labels = dti_df[dti_cfg["label_col"]]

train_df, val_df, train_labels, val_labels = train_test_split(dti_df, labels,stratify=labels, test_size=0.2, random_state=config["runtime"]["seed"])


In [160]:
train_df.shape

(27792, 7)

In [161]:
test_df.shape

(3957, 7)

In [162]:
train_df[dti_cfg["label_col"]].value_counts()

label
0    18005
1     9787
Name: count, dtype: int64

In [163]:
# check labels in train df



# check labels in val df

val_df[dti_cfg["label_col"]].value_counts()


label
0    4502
1    2447
Name: count, dtype: int64

In [165]:
from torch.utils.data import DataLoader

device = config["runtime"]["device"]
pos_weight = compute_pos_weight(train_df, config["datasets"]["dti"]["label_col"])

train_ds = NegativeSamplingDTIDataset(train_df, config, emb_reg, neg_per_pos=3, seed=config["runtime"]["seed"])
val_ds   = DTIDataset(val_df, config, emb_reg)  # your earlier “raw” dataset is fine for val

train_loader = DataLoader(
    train_ds,
    batch_size=512,
    shuffle=True,
    collate_fn=collate_dti
)

val_loader = DataLoader(
    val_ds,
    batch_size=512,
    shuffle=False,
    collate_fn=collate_dti
)

model = DTIScorer(
    config,
    drug_in_dims=[emb_reg.drug[n].dim for n in config["fusion"]["drug"]["embed"]],
    prot_in_dims=[emb_reg.protein[n].dim for n in config["fusion"]["protein"]["embed"]],
).to(device)

criterion = DTISupervisedContrastiveLoss(
    temperature=0.07,
    pos_weight=1.5
).to(device)


opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

for epoch in range(1, 25):
    tr = train_one_epoch(model, train_loader, opt, device, criterion, grad_clip=1.0, use_amp=True)
    auc = eval_aucroc_explicit(model, val_loader, device)
    print(f"epoch {epoch} | train_loss={tr:.4f} | val_aucroc={auc:.4f}")


epoch 1 | train_loss=0.8880 | val_aucroc=0.4843
epoch 2 | train_loss=0.6026 | val_aucroc=0.7568
epoch 3 | train_loss=0.5378 | val_aucroc=0.6754
epoch 4 | train_loss=0.5278 | val_aucroc=0.6228
epoch 5 | train_loss=0.5312 | val_aucroc=0.6184
epoch 6 | train_loss=0.5039 | val_aucroc=0.5818
epoch 7 | train_loss=0.5054 | val_aucroc=0.4003
epoch 8 | train_loss=0.4932 | val_aucroc=0.3067
epoch 9 | train_loss=0.4920 | val_aucroc=0.4460
epoch 10 | train_loss=0.4817 | val_aucroc=0.3373
epoch 11 | train_loss=0.5206 | val_aucroc=0.3563
epoch 12 | train_loss=0.4813 | val_aucroc=0.2788
epoch 13 | train_loss=0.4694 | val_aucroc=0.2520
epoch 14 | train_loss=0.4833 | val_aucroc=0.2863
epoch 15 | train_loss=0.4760 | val_aucroc=0.3771
epoch 16 | train_loss=0.4381 | val_aucroc=0.3323
epoch 17 | train_loss=0.4388 | val_aucroc=0.5013
epoch 18 | train_loss=0.4239 | val_aucroc=0.4849
epoch 19 | train_loss=0.4449 | val_aucroc=0.3149
epoch 20 | train_loss=0.4520 | val_aucroc=0.3515
epoch 21 | train_loss=0.4240 

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda
